# Intelia Executive Intelligence Canvas
Live exploration workspace for CCO, CPO, and CTO board meeting preparation. PII-safe — customer cells use dim_customers_analyst (masked view).

### Revenue Trend — Last 12 Months
**Audience:** CCO

**Visualization:** Line Chart

*Click 'Explain with Gemini' to get an AI summary of the revenue trend. Source: gold.mart_revenue_summary*

In [ ]:
%%bigquery
SELECT FORMAT_DATE('%Y-%m', summary_date) AS month, SUM(gross_revenue) AS revenue, SUM(order_count) AS orders, SUM(new_customers) AS new_customers, SUM(repeat_customers) AS repeat_customers FROM `vishal-sandpit-474523.gold.mart_revenue_summary` WHERE summary_date >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 12 MONTH), MONTH) GROUP BY 1 ORDER BY 1

### Revenue by Country — Last 3 Months
**Audience:** CCO, CPO

**Visualization:** Bar Chart

*Geographic revenue breakdown. Source: gold.mart_revenue_summary*

In [ ]:
%%bigquery
SELECT country, SUM(gross_revenue) AS total_revenue, SUM(order_count) AS orders, SUM(unique_customers) AS customers, ROUND(SUM(gross_revenue) / NULLIF(SUM(unique_customers), 0), 2) AS revenue_per_customer FROM `vishal-sandpit-474523.gold.mart_revenue_summary` WHERE summary_date >= DATE_TRUNC(DATE_SUB(CURRENT_DATE(), INTERVAL 3 MONTH), MONTH) GROUP BY country ORDER BY total_revenue DESC

### Customer Segment & Churn Risk (PII-safe)
**Audience:** CCO

**Visualization:** Table

*Uses analyst-safe view — no raw PII. LTV shown as band (0-999, 1000-4999 etc). Source: gold.dim_customers_analyst*

In [ ]:
%%bigquery
SELECT customer_segment, churn_risk, COUNT(*) AS customers, ROUND(AVG(order_count), 1) AS avg_orders, ROUND(AVG(days_since_last_purchase)) AS avg_days_inactive FROM `vishal-sandpit-474523.gold.dim_customers_analyst` GROUP BY customer_segment, churn_risk ORDER BY customer_segment, churn_risk

### Gemini Retention Insights — At Risk Customers
**Audience:** CCO, CPO

**Visualization:** Table

*Top at-risk customers with Gemini retention strategies. Filter by segment to narrow focus. Sources: ai.customer_concierge + gold.dim_customers_analyst*

In [ ]:
%%bigquery
SELECT c.customer_id, c.customer_segment, c.days_since_last_purchase, c.lifetime_value_band, ai.gemini_insight FROM `vishal-sandpit-474523.gold.dim_customers_analyst` c JOIN `vishal-sandpit-474523.ai.customer_concierge` ai USING (customer_id) WHERE c.churn_risk IN ('At Risk', 'Cooling') ORDER BY c.days_since_last_purchase DESC LIMIT 20

### Top Products by Revenue — Last 90 Days
**Audience:** CPO

**Visualization:** Bar Chart

*Top 15 products. Note: unit_price column is tagged Sensitive Financial — CPO access required. Source: gold.dim_products + silver.stg_order_items*

In [ ]:
%%bigquery
SELECT p.product_name, p.category, COUNT(DISTINCT o.order_id) AS order_count, SUM(oi.subtotal) AS total_revenue, ROUND(AVG(oi.unit_price), 2) AS avg_selling_price FROM `vishal-sandpit-474523.silver.stg_order_items` oi JOIN `vishal-sandpit-474523.silver.stg_orders` o USING (order_id) JOIN `vishal-sandpit-474523.gold.dim_products` p USING (product_id) WHERE o.order_date >= DATE_SUB(CURRENT_DATE(), INTERVAL 90 DAY) GROUP BY p.product_name, p.category ORDER BY total_revenue DESC LIMIT 15

### Gemini Product Upsell Strategies
**Audience:** CPO

**Visualization:** Table

*Gemini-generated cross-sell and upsell strategies per product. Source: ai.product_upsell*

In [ ]:
%%bigquery
SELECT p.product_name, p.category, pu.gemini_upsell_insight FROM `vishal-sandpit-474523.ai.product_upsell` pu JOIN `vishal-sandpit-474523.gold.dim_products` p USING (product_id) WHERE p.is_active = TRUE LIMIT 10

### Data Pipeline Health — Last 7 Days
**Audience:** CTO

**Visualization:** Table

*Delta pipeline runs by entity. Any 'FAILED' rows need investigation in Cloud Workflows console. Source: governance.batch_audit_log*

In [ ]:
%%bigquery
SELECT entity, status, COUNT(*) AS run_count, MAX(rows_merged) AS max_rows_merged, MAX(load_timestamp) AS last_run FROM `vishal-sandpit-474523.governance.batch_audit_log` WHERE load_timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY) GROUP BY entity, status ORDER BY entity, status

### Business Glossary
**Audience:** ALL

**Visualization:** Table

*Authoritative definitions for all key warehouse terms. Source: governance.business_glossary*

In [ ]:
%%bigquery
SELECT term, definition, domain, owner, update_frequency, related_tables FROM `vishal-sandpit-474523.governance.business_glossary` ORDER BY domain, term